# Workflow Engines: Snakemake and Nextflow

**Tier 3 -- Applied Bioinformatics**

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain **why** workflow engines are essential for reproducible bioinformatics
2. Write **Snakemake** rules with inputs, outputs, wildcards, and shell commands
3. Understand **Nextflow** processes, channels, and dataflow programming
4. Configure workflows for **parallelization** and **container** execution
5. Choose the appropriate workflow engine for different use cases
6. Use **nf-core** pipelines for standard genomics analyses

---

---

[← Previous: Single-Cell Analysis](./01_single_cell_scanpy.ipynb) | [Next: Testing and CI/CD →](./03_testing_cicd.ipynb)

---

## 1. Why Workflow Engines?

### The Problem with Shell Scripts

A typical bioinformatics pipeline involves many steps:

```bash
# Quality control
fastqc sample1.fastq.gz -o qc/

# Trimming
trimmomatic PE sample1_R1.fastq.gz sample1_R2.fastq.gz ...

# Alignment
bwa mem reference.fa sample1_R1.trimmed.fastq.gz sample1_R2.trimmed.fastq.gz > sample1.sam

# ... repeat for 100 samples
```

**Problems emerge at scale:**

| Problem | Impact |
|---------|--------|
| No parallelization | 100 samples run sequentially -- days instead of hours |
| No dependency tracking | Change one file → must re-run everything manually |
| No checkpointing | Pipeline fails at step 50 → restart from scratch |
| Environment inconsistency | Works on your laptop, fails on the cluster |
| No provenance | Which parameters produced this BAM file? |

### The Solution: Workflow Engines

**Workflow engines** solve these problems by:

1. **Dependency resolution**: Only re-run steps whose inputs changed
2. **Automatic parallelization**: Run independent steps simultaneously
3. **Portability**: Execute the same workflow locally, on HPC, or in the cloud
4. **Containerization**: Package tools in Docker/Singularity for reproducibility
5. **Provenance tracking**: Record exactly what ran and when

---

## 2. Snakemake

**Snakemake** is a Python-based workflow engine using Make-like syntax. It's excellent for:

- Python-native bioinformaticians
- File-based workflows (input files → output files)
- HPC cluster environments

### Core Concepts

| Concept | Description | Example |
|---------|-------------|----------|
| **Rule** | A processing step | `rule fastqc:` |
| **Input** | Required files | `input: "data/{sample}.fastq.gz"` |
| **Output** | Produced files | `output: "qc/{sample}_fastqc.html"` |
| **Wildcard** | Pattern variable | `{sample}` matches any string |
| **Shell** | Command to run | `shell: "fastqc {input}"` |

In [ ]:
# Example Snakefile for RNA-seq analysis
snakefile_content = '''
# ============================================
# RNA-seq Pipeline with Snakemake
# ============================================

# Configuration
SAMPLES = ["sample1", "sample2", "sample3"]
GENOME = "reference/genome.fa"
GTF = "reference/genes.gtf"

# Target rule: defines the final outputs we want
rule all:
    input:
        "results/counts_matrix.csv",
        expand("qc/{sample}_fastqc.html", sample=SAMPLES)

# Rule 1: Quality control with FastQC
rule fastqc:
    input:
        "data/{sample}.fastq.gz"
    output:
        html = "qc/{sample}_fastqc.html",
        zip = "qc/{sample}_fastqc.zip"
    threads: 2
    shell:
        "fastqc {input} -o qc/ -t {threads}"

# Rule 2: Alignment with STAR
rule align:
    input:
        fastq = "data/{sample}.fastq.gz",
        index = "reference/star_index"
    output:
        bam = "aligned/{sample}.bam"
    threads: 8
    resources:
        mem_mb = 32000  # 32 GB RAM
    shell:
        """
        STAR --genomeDir {input.index} \
             --readFilesIn {input.fastq} \
             --readFilesCommand zcat \
             --runThreadN {threads} \
             --outSAMtype BAM SortedByCoordinate \
             --outFileNamePrefix aligned/{wildcards.sample}_
        mv aligned/{wildcards.sample}_Aligned.sortedByCoord.out.bam {output.bam}
        """

# Rule 3: Count reads per gene
rule count:
    input:
        bam = "aligned/{sample}.bam",
        gtf = GTF
    output:
        "counts/{sample}.txt"
    threads: 4
    shell:
        "featureCounts -T {threads} -a {input.gtf} -o {output} {input.bam}"

# Rule 4: Merge counts into matrix
rule merge_counts:
    input:
        expand("counts/{sample}.txt", sample=SAMPLES)
    output:
        "results/counts_matrix.csv"
    script:
        "scripts/merge_counts.py"
'''

print(snakefile_content)

### How Snakemake Works

Snakemake uses **backward chaining**:

1. Start from the target outputs (rule `all`)
2. Find rules that produce those outputs
3. Check what inputs those rules need
4. Recursively build a **Directed Acyclic Graph (DAG)** of jobs
5. Execute jobs respecting dependencies

```
                    counts_matrix.csv
                           |
            +--------------+---------------+
            |              |               |
     sample1.txt     sample2.txt     sample3.txt
            |              |               |
     sample1.bam     sample2.bam     sample3.bam
            |              |               |
     sample1.fastq   sample2.fastq   sample3.fastq
```

In [ ]:
# Running Snakemake from the command line
commands = '''
# Dry run: see what would execute without running
snakemake -n

# Run with 8 CPU cores
snakemake --cores 8

# Run with conda environments for each rule
snakemake --cores 8 --use-conda

# Visualize the job DAG
snakemake --dag | dot -Tpng > dag.png

# Run on SLURM cluster (100 parallel jobs)
snakemake --cluster "sbatch -p normal -c {threads} --mem {resources.mem_mb}" --jobs 100

# Force re-run a specific rule
snakemake --forcerun align --cores 8
'''
print(commands)

---

## 3. Nextflow

**Nextflow** uses a dataflow programming model where data flows through **channels** into **processes**. It excels at:

- Cloud-native workflows (AWS, Google Cloud, Azure)
- Container-first design (Docker, Singularity built-in)
- Complex branching/merging dataflows

### Core Concepts

| Concept | Description | Example |
|---------|-------------|----------|
| **Channel** | Data stream | `Channel.fromFilePairs()` |
| **Process** | Computation unit | `process FASTQC { ... }` |
| **Workflow** | Connects processes | `workflow { ... }` |
| **Directive** | Process config | `cpus 8`, `memory '32 GB'` |

In [ ]:
# Example Nextflow pipeline (main.nf)
nextflow_content = '''
#!/usr/bin/env nextflow
nextflow.enable.dsl=2

// Pipeline parameters (can be overridden at runtime)
params.reads = "data/*_{1,2}.fastq.gz"
params.genome = "reference/genome.fa"
params.outdir = "results"

// Create a channel from paired-end reads
Channel
    .fromFilePairs(params.reads, checkIfExists: true)
    .set { read_pairs_ch }

// Process 1: Quality Control
process FASTQC {
    tag "$sample_id"                      // Label in logs
    publishDir "${params.outdir}/qc"      // Copy outputs here
    cpus 2
    memory '4 GB'
    
    container 'biocontainers/fastqc:v0.11.9'  // Docker container
    
    input:
    tuple val(sample_id), path(reads)
    
    output:
    path "*_fastqc.{html,zip}"
    
    script:
    """
    fastqc ${reads} --threads ${task.cpus}
    """
}

// Process 2: Alignment
process ALIGN {
    tag "$sample_id"
    publishDir "${params.outdir}/aligned", mode: 'copy'
    cpus 8
    memory '32 GB'
    
    container 'biocontainers/bwa:v0.7.17'
    
    input:
    tuple val(sample_id), path(reads)
    path genome
    
    output:
    tuple val(sample_id), path("${sample_id}.bam")
    
    script:
    """
    bwa mem -t ${task.cpus} ${genome} ${reads} | \
        samtools sort -@ ${task.cpus} -o ${sample_id}.bam
    samtools index ${sample_id}.bam
    """
}

// Workflow definition: connect processes
workflow {
    // QC runs in parallel with alignment
    FASTQC(read_pairs_ch)
    ALIGN(read_pairs_ch, params.genome)
}
'''

print(nextflow_content)

In [ ]:
# Running Nextflow from the command line
commands = '''
# Run pipeline locally
nextflow run main.nf

# Override parameters
nextflow run main.nf --reads "mydata/*.fastq.gz" --outdir my_results

# Run with Docker containers
nextflow run main.nf -with-docker

# Run with Singularity (for HPC clusters)
nextflow run main.nf -with-singularity

# Resume a failed/interrupted run
nextflow run main.nf -resume

# Run an nf-core pipeline directly from GitHub
nextflow run nf-core/rnaseq -r 3.12.0 \
    --input samplesheet.csv \
    --genome GRCh38 \
    -profile docker
'''
print(commands)

---

## 4. nf-core: Community Pipelines

**nf-core** is a community effort providing:

- **Pre-built pipelines** for common analyses (RNA-seq, ChIP-seq, variant calling, etc.)
- **Best practices** for Nextflow development
- **Standardized** inputs, outputs, and reporting

### Popular nf-core Pipelines

| Pipeline | Application |
|----------|-------------|
| `nf-core/rnaseq` | Bulk RNA-seq quantification and DE |
| `nf-core/sarek` | Germline/somatic variant calling |
| `nf-core/chipseq` | ChIP-seq peak calling |
| `nf-core/atacseq` | ATAC-seq analysis |
| `nf-core/ampliseq` | 16S amplicon analysis |
| `nf-core/scrnaseq` | Single-cell RNA-seq |

---

## 5. Comparison: Snakemake vs Nextflow

| Feature | Snakemake | Nextflow |
|---------|-----------|----------|
| **Language** | Python-like | Groovy DSL |
| **Paradigm** | File-based (Make) | Dataflow (channels) |
| **Learning curve** | Easier (Python users) | Steeper |
| **Cloud support** | Good (via plugins) | Excellent (native) |
| **Container integration** | Good | Excellent |
| **Community pipelines** | Snakemake catalog | nf-core (larger) |
| **HPC support** | Excellent | Excellent |
| **Best for** | Custom pipelines, Python shops | Production genomics, cloud |

### When to Use Each

**Choose Snakemake if:**
- You're comfortable with Python
- You need tight integration with Python analysis code
- You're building a custom, one-off pipeline

**Choose Nextflow if:**
- You need production-grade genomics pipelines
- You want to leverage nf-core community pipelines
- You're running on cloud infrastructure (AWS, GCP)

---

## Exercises

### Exercise 1: Design a Snakemake Workflow

Write Snakemake rules for a variant calling pipeline:
1. FastQC → 2. Trim (fastp) → 3. Align (BWA) → 4. Mark duplicates (Picard) → 5. Call variants (bcftools)

In [ ]:
# Your Snakefile here
# rule fastqc:
#     input: "data/{sample}.fastq.gz"
#     output: "qc/{sample}_fastqc.html"
#     shell: "fastqc {input} -o qc/"
#
# rule trim:
#     ...

### Exercise 2: Add Conda Environment

Create a `envs/fastqc.yaml` file and modify the fastqc rule to use it with `conda:` directive.

In [ ]:
# envs/fastqc.yaml content:
# channels:
#   - bioconda
#   - conda-forge
# dependencies:
#   - fastqc=0.11.9
#
# Modified rule:
# rule fastqc:
#     conda: "envs/fastqc.yaml"
#     ...

### Exercise 3: Run an nf-core Pipeline

Using the nf-core website, find the command to run `nf-core/fetchngs` to download data from the SRA.

In [ ]:
# Your command here
# Hint: nextflow run nf-core/fetchngs ...

---

## Summary

| Concept | Snakemake | Nextflow |
|---------|-----------|----------|
| **Define a step** | `rule name:` | `process NAME { }` |
| **Inputs** | `input: "path/{wildcard}.txt"` | `input: path(file)` |
| **Outputs** | `output: "path/{wildcard}.out"` | `output: path("*.out")` |
| **Run command** | `shell: "cmd"` | `script: """cmd"""` |
| **Parallelism** | `threads: N` | `cpus N` |
| **Container** | `container: "url"` or `singularity:` | `container 'image'` |
| **Dry run** | `snakemake -n` | N/A (uses `-resume`) |
| **Execute** | `snakemake --cores N` | `nextflow run main.nf` |

---

---

[← Previous: Single-Cell Analysis](./01_single_cell_scanpy.ipynb) | [Next: Testing and CI/CD →](./03_testing_cicd.ipynb)

---